# Klasifikasi Toksisitas IndoToxic2024 dengan IndoBERT

Alur:
1. Muat data train, validasi, dan test (70:15:15)
2. Normalisasi kata alay dengan `new_kamusalay.csv` (opsional)
3. Fine-tuning IndoBERT
4. Evaluasi di data uji: Macro-F1 (utama), Precision, Recall, dan Confusion Matrix

Notebook ini untuk VS Code (Jupyter extension), tanpa Google Colab.

Cara pakai:
1. Pilih kernel Python di pojok kanan atas VS Code.
2. Jalankan cell dari atas ke bawah.
3. Cek output cell konfigurasi. Notebook mengatur parameter otomatis untuk GPU atau CPU.

Untuk GPU NVIDIA di Windows, install PyTorch versi CUDA lewat terminal (bukan lewat notebook):
`pip install torch --index-url https://download.pytorch.org/whl/cu121`
Tanpa GPU, notebook memakai mode CPU yang lebih ringan.

In [ ]:
%pip install -q transformers datasets accelerate scikit-learn seaborn matplotlib

## 1. Konfigurasi

In [ ]:
import os, random, inspect
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
from pathlib import Path
import numpy as np
import pandas as pd
import torch

BASE_DIR      = Path(r"C:\Users\amel\Documents\IndoToxic")
DATA_DIR      = BASE_DIR / "data" / "processed"
KAMUS_PATH    = BASE_DIR / "new_kamusalay.csv"
OUTPUT_DIR    = BASE_DIR / "hasil_indobert"

MODEL_NAME    = "indobenchmark/indobert-base-p1"
LR            = 2e-5
SEED          = 42

# Untuk uji cepat di CPU, isi misalnya 2000. Biarkan None untuk memakai semua data train.
TRAIN_SUBSET  = None
# True jika ingin melanjutkan training dari checkpoint terakhir setelah terhenti
RESUME        = False

TEXT_COL      = "text_clean"   # teks hasil preprocessing
LABEL_COL     = "toxicity"     # label biner di file jsonl (0 = tidak toksik, 1 = toksik)

# Setel False jika text_clean sudah dinormalisasi dengan kamusalay
USE_KAMUSALAY = True

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(SEED)
HAS_GPU = torch.cuda.is_available()
if HAS_GPU:
    print("GPU terdeteksi:", torch.cuda.get_device_name(0))
    MAX_LEN, EPOCHS, BATCH_SIZE = 128, 4, 32
else:
    print("GPU tidak terdeteksi. Memakai mode CPU (parameter lebih ringan).")
    MAX_LEN, EPOCHS, BATCH_SIZE = 64, 2, 16
print("MAX_LEN, EPOCHS, BATCH_SIZE =", MAX_LEN, EPOCHS, BATCH_SIZE)

## 2. Muat data

In [ ]:
def load(name):
    df = pd.read_json(DATA_DIR / name, lines=True)
    for c in (TEXT_COL, LABEL_COL):
        assert c in df.columns, f"Kolom '{c}' tidak ada di {name}. Kolom tersedia: {df.columns.tolist()}"
    # Ambil kolom yang dibutuhkan saja. Kolom lain (misal 'fitur' berisi dict) diabaikan.
    df = df[[TEXT_COL, LABEL_COL]].copy()
    df.columns = ["text", "label_asli"]
    df = df.dropna(subset=["text", "label_asli"]).reset_index(drop=True)
    df["text"] = df["text"].astype(str)
    # Jika label terbaca sebagai float (misal 1.0), ubah ke int
    if pd.api.types.is_numeric_dtype(df["label_asli"]) and (df["label_asli"] % 1 == 0).all():
        df["label_asli"] = df["label_asli"].astype(int)
    return df

train_df = load("train_preprocessed.jsonl")
val_df   = load("val_preprocessed.jsonl")
test_df  = load("test_preprocessed.jsonl")

if TRAIN_SUBSET:
    train_df = train_df.sample(n=min(TRAIN_SUBSET, len(train_df)), random_state=SEED).reset_index(drop=True)

print("Ukuran train/val/test:", len(train_df), len(val_df), len(test_df))
train_df.head()

In [ ]:
# Buat label 0..K-1 dari gabungan semua split
semua    = pd.concat([train_df["label_asli"], val_df["label_asli"], test_df["label_asli"]])
classes  = sorted(semua.unique().tolist())
label2id = {c: i for i, c in enumerate(classes)}
id2label = {i: str(c) for c, i in label2id.items()}
NUM_LABELS = len(classes)
print("Label:", label2id)

for d in (train_df, val_df, test_df):
    d["labels"] = d["label_asli"].map(label2id).astype("int64")

print()
for nama, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(nama, d["labels"].value_counts(normalize=True).sort_index().round(3).to_dict())

## 3. Normalisasi kata alay (kamusalay)

In [ ]:
kamus = pd.read_csv(KAMUS_PATH, header=None, names=["alay", "baku"], encoding="latin-1")
kamus = kamus.dropna()
kamus_dict = dict(zip(kamus["alay"].astype(str).str.lower(), kamus["baku"].astype(str).str.lower()))
print("Jumlah entri kamus:", len(kamus_dict))

def normalisasi(teks):
    return " ".join(kamus_dict.get(w, w) for w in str(teks).lower().split())

if USE_KAMUSALAY:
    for d in (train_df, val_df, test_df):
        d["text"] = d["text"].apply(normalisasi)

train_df[["text", "labels"]].head()

## 4. Tokenisasi

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_ds(df):
    ds = Dataset.from_pandas(df[["text", "labels"]].reset_index(drop=True), preserve_index=False)
    return ds.map(lambda b: tokenizer(b["text"], truncation=True, max_length=MAX_LEN),
                  batched=True, remove_columns=["text"])

train_ds, val_ds, test_ds = to_ds(train_df), to_ds(val_df), to_ds(test_df)
collator = DataCollatorWithPadding(tokenizer)
print(train_ds)

## 5. Fine-tuning IndoBERT

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import (AutoModelForSequenceClassification, TrainingArguments,
                          Trainer, EarlyStoppingCallback)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=NUM_LABELS, id2label=id2label,
    label2id={v: k for k, v in id2label.items()})

def compute_metrics(eval_pred):
    logits, y_true = eval_pred
    y_pred = np.argmax(logits, axis=-1)
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    return {"accuracy": accuracy_score(y_true, y_pred),
            "precision_macro": p, "recall_macro": r, "f1_macro": f1}

# Kompatibel dengan versi transformers lama dan baru
ta_params = inspect.signature(TrainingArguments.__init__).parameters
eval_key  = "eval_strategy" if "eval_strategy" in ta_params else "evaluation_strategy"

ta_kwargs = dict(
    output_dir=str(OUTPUT_DIR / "ckpt"),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=64 if HAS_GPU else 32,
    learning_rate=LR,
    weight_decay=0.01,
    warmup_ratio=0.1,
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    save_total_limit=1,
    fp16=HAS_GPU,
    seed=SEED,
    report_to="none",
)
ta_kwargs[eval_key] = "epoch"
if "group_by_length" in ta_params:
    ta_kwargs["group_by_length"] = True      # mempercepat training di CPU
if "dataloader_num_workers" in ta_params:
    ta_kwargs["dataloader_num_workers"] = 0  # aman di Windows
args = TrainingArguments(**ta_kwargs)

tr_params = inspect.signature(Trainer.__init__).parameters
tok_key   = "processing_class" if "processing_class" in tr_params else "tokenizer"

trainer = Trainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=val_ds,
    data_collator=collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    **{tok_key: tokenizer},
)
ckpt_dir = OUTPUT_DIR / "ckpt"
ada_ckpt = ckpt_dir.exists() and any(ckpt_dir.glob("checkpoint-*"))
trainer.train(resume_from_checkpoint=True if (RESUME and ada_ckpt) else None)

## 6. Evaluasi

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

ids   = list(range(NUM_LABELS))
names = [id2label[i] for i in ids]

def evaluasi(ds, nama):
    out = trainer.predict(ds)
    y_true = out.label_ids
    y_pred = np.argmax(out.predictions, axis=-1)
    print(f"===== {nama} =====")
    print(classification_report(y_true, y_pred, labels=ids, target_names=names,
                                digits=4, zero_division=0))
    return y_true, y_pred

yv_true, yv_pred = evaluasi(val_ds, "VALIDASI")
yt_true, yt_pred = evaluasi(test_ds, "TEST")

In [ ]:
p, r, f1, _ = precision_recall_fscore_support(yt_true, yt_pred, average="macro", zero_division=0)
ringkasan = pd.DataFrame({
    "Metrik": ["Macro-F1 (utama)", "Precision (macro)", "Recall (macro)", "Accuracy"],
    "Nilai":  [f1, p, r, accuracy_score(yt_true, yt_pred)],
}).round(4)
print(ringkasan.to_string(index=False))

per_kelas = pd.DataFrame(classification_report(
    yt_true, yt_pred, labels=ids, target_names=names,
    output_dict=True, zero_division=0)).T.round(4)
print()
print(per_kelas.to_string())

In [ ]:
cm  = confusion_matrix(yt_true, yt_pred, labels=ids)
cmn = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)

fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=names, yticklabels=names, ax=ax[0])
ax[0].set_title("Confusion Matrix (jumlah)")
sns.heatmap(cmn, annot=True, fmt=".2%", cmap="Blues", xticklabels=names, yticklabels=names, ax=ax[1])
ax[1].set_title("Confusion Matrix (normalisasi)")
for a in ax:
    a.set_xlabel("Prediksi"); a.set_ylabel("Aktual")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "confusion_matrix.png", dpi=150)
plt.show()

if NUM_LABELS == 2:
    tn, fp, fn, tp = cm.ravel()
    print(f"TP={tp}  FP={fp}  FN={fn}  TN={tn}")

In [ ]:
ringkasan.to_csv(OUTPUT_DIR / "ringkasan_metrik_test.csv", index=False)
per_kelas.to_csv(OUTPUT_DIR / "metrik_per_kelas_test.csv")
pd.DataFrame({"aktual": yt_true, "prediksi": yt_pred}).to_csv(OUTPUT_DIR / "prediksi_test.csv", index=False)
trainer.save_model(str(OUTPUT_DIR / "model_final"))
tokenizer.save_pretrained(str(OUTPUT_DIR / "model_final"))
print("Tersimpan di:", OUTPUT_DIR)

In [ ]:
# Uji cepat satu kalimat
model.eval()
def prediksi(teks):
    if USE_KAMUSALAY:
        teks = normalisasi(teks)
    enc = tokenizer(teks, return_tensors="pt", truncation=True, max_length=MAX_LEN).to(model.device)
    with torch.no_grad():
        prob = torch.softmax(model(**enc).logits, dim=-1)[0].cpu().numpy()
    return {id2label[i]: round(float(prob[i]), 4) for i in range(NUM_LABELS)}

print(prediksi("kamu bodoh sekali"))